# Plotting data
There is significant functionality built in to {py:mod}`xarray` for [plotting data](https://docs.xarray.dev/en/stable/user-guide/plotting.html). `peaks` leverages this, as well as adding some additional specific functionality.

In [ ]:
# Import packages
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import peaks as pks
import os

# Set default options
xr.set_options(cmap_sequential='Purples', keep_attrs=True)
%matplotlib inline
%config InlineBackend.figure_format='retina'

# For rendering output from iplot
from bokeh.io import output_notebook
output_notebook()

:::{tip}
The data used in this tutorial is obtained using taken from the {py:mod}`sample_data <peaks.core.utils.sample_data>` module. This is downloaded as required. In normal use, load data as outlined in the [Getting Started](./1_getting_started.ipynb#loading-data) guide.
:::

In [ ]:
from peaks.core.utils.sample_data import ExampleData, plot_tutorial_example_figure

## Static plots
For a simple dispersion, access via the in-built {py:meth}`.plot <xarray.DataArray.plot>` method:

In [ ]:
disp1 = ExampleData.dispersion()

In [ ]:
disp1.plot()

### Grid plots
Often, it is convenient to plot a set of dispersions in some form of grid plot. We have a convenience method {py:func}`plot_grid <peaks.core.display.plotting.plot_grid>` for this, which accepts a list of the (1D or 2D) dataarrays to plot, as well as various other plotting options, e.g.:

In [ ]:
disp_list = [disp1,disp1*2*pks.ureg('count/s'),disp1+1000*pks.ureg('count/s'),disp1-1000*pks.ureg('count/s')]
labels = ['original','two_x','add_1000','minus_1000']
pks.plot_grid(disp_list,
             titles=labels,
             ncols=4,
             sharey=True,
             cmap=['Blues','Greys',None,'Purples'],
             vmax=[None,None,1002,-999])

Alternatively, a grid plot of 1D or 2D data can be generated by calling {py:func}`.plot_grid() <peaks.core.display.plotting.plot_grid>` directly on data stored in the {py:class}`xarray.DataTree` format, see [grouping scans](#./1_getting_started.ipynb#grouping_scans). In this case, the tree should be hollow, and the leaves should be {py:class}`xarray.Dataset`'s with a single data variable, `data`, as is created using the `peaks` {py:func}`.add() <peaks.core.utils.datatree_utils.add>` method:

In [ ]:
# Make a blank datatree
dt = xr.DataTree()
# Populate it with the scans from above
for disp, name in zip(disp_list,labels):
    dt.add(disp, name)

# Plot the dispersions
dt.plot_grid(ncols=4, sharey=True, add_colorbar=False)

### Plotting higher-dimensional data

If the {py:class}`xarray.DataArray` itself is higher dimensional, the [in-built `xarray` methods](https://docs.xarray.dev/en/latest/user-guide/plotting.html#faceting) can be used to generate grid-type plots. E.g. let's load a spatial map:

In [ ]:
SM1 = ExampleData.SM()

We can make a {py:class}`facetgrid <xarray.plot.FacetGrid>` plot of this, with the spatial co-ordinates as the grid axes. Note, however, the above dataset has $31\times14$ spatial points, which would be too many to plot, so we'll first pre-select a few datapoints from the grid.

In [ ]:
SM1.isel(x1=slice(0,None,3),x2=slice(0,None,6)).plot(row='x1',col='x2', robust=True)

Note the `robust=True` argument as a quick way to enhance contrast when there are outliers.

### Stack plots
A waterfall stackplot can be made using {py:func}`.plot_DCs <peaks.core.display.plotting.plot_DCs>`, passing a list of 1D {py:class}`xarray.DataArray`s, a single {py:class}`xarray.DataArray` with the stack of DCs included, or a {py:class}`xarray.DataTree` of DCs.

In [ ]:
EDCs_to_plot = disp1.EDC((0,10,1), 1)  # Select a set of EDCs to plot from the dispersion
EDCs_to_plot.plot_DCs(
             offset=0.1,  # offset the plots by 10% of the maximum peak height
            )  # Plot EDCs

In [ ]:
# Make a blank datatree
dt = xr.DataTree()
# Populate it with EDCs
for i in range(10):
    EDC = disp1.EDC(i, 1)
    dt.add(EDC, f"EDC_{i}")

# Plot the dispersions
dt.plot_DCs(offset=0.1)

### FS stack plots
A Fermi surface stack plot can also be created by passing a single {py:class}`xarray.DataArray` containing the cuts to plot

In [ ]:
# Load Fermi surface
FS1 = ExampleData.FS()

# Extract a stack of the relevant slices
FS_stack = FS1.sel(theta_par=slice(-12,12), polar=slice(-10,14)).MDC((104.86,105.06,0.025),0.01)

In [ ]:
FS_stack.plot_3d_stack(vmax=8, aspect=(1,1,200))

## Interactive plotting
### Interactive UIs
Interactive UIs for visualising and aligning data can be accessed by calling {py:func}`.disp <peaks.core.GUI.disp_panels.disp.disp>` on a 2D, 3D, or 4D {py:class}`xarray.DataArray`, or on a {py:class}`xarray.DataTree` containing  either a single data entry, or on a tree with multiple leafs. If the latter, or if passing a list of 2D {py:class}`xarray.DataArray`'s to {py:func}`pks.disp <peaks.core.GUI.disp_panels.disp.disp>`, all 2D scans are included in a data explorer view.
```python
# Show an explorer view of multiple 2D DataArrays stored in a DataTree `dt`
dt.disp()  # Open the viewer
```

#### 2D display panel

In [ ]:
# Static view of the Display Panel for the documentation
plot_tutorial_example_figure('2D_disp_ex.png', max_width="70%")

**Hints:**
- Select different scans using the box on the bottom right
- Scroll/drag with the mouse on the plots or their axes to zoom/pan
- The following keyboard shortcuts are defined:
    - Line cuts can be selected by dragging the cursors or by using the shortcuts:
        - `Arrow keys` to move crosshair 0
        - `Control` (`Command` on Mac) + `Arrow keys` to move crosshair 1
        - `Shift` + `Arrow keys` to move all currently shown crosshairs
    - Hold down `Space` to temporarily hide all crosshairs
    - `0`, `1`, `2` to toggle visibliity of the crosshairs
    - `M` to enable/disable mirrored mode, where a mirrored version of the data is shown for axes where centering is supported
- Click `Align` to attempt an automatic centering based on the field of view shown in the mail panel
- Click `Copy` to copy the normal emission parameters corresponding to crosshair 0 to the clipboard, in a form suitable for applying to the data
- The zoom range and colour scale can be locked or not when changing scans using the `Lock` checkboxes
- Right click on the colour bar to change colour scales

In addition, some options are included in calling {py:func}`disp <peaks.core.GUI.disp_panels.disp.disp>` for customising the display, in particular `primary_dim` which can be used to help set the order of axes on the live displays, and `exclude_from_centering` for overriding auto guestimates on which dimensions to use for centering. 

#### 3D display panel
A 3D explorer works in much the same way:

```python
FS1.disp()  # Directly show a 3D dataarray
```

In [ ]:
# Static view of the Display Panel for the documentation
plot_tutorial_example_figure("3D_disp_ex.png")

#### 4D display panel
4D data can be displayed in the same way.

:::{tip}
4D data can be very heavy, making it memory intensive for running the UI. It is strongly recommended to first bin your data and crop to the relevant data range. The UI can handle being passed lazy [`dask`](https://docs.dask.org/en/stable/)-backed arrays. **If** the data will fit in available memory, a significant speed up in live visualisation is obtained if the data is first persisted:
```python
SM1.sel(theta_par=slice(-18,18),eV=slice(73.68,76.75))  # Crop to relevant range 
    .bin_data(2)  # 2x2 binning
    .chunk(x1=1,x2=1,eV=9999,theta_par=9999)  # Ensure chunks are aligned with spatial dims
    .persist()  # Load data into memory
    .disp()  # Load display panel
```
Alternatively, you can use {py:meth}`.compute() <xarray.DataArray.compute>` to load the data as a pure {py:mod}`numpy`-backed array, but this can make some of the ROI data selections slower.
:::

The 4D display has two tabs:
- A simple explorer tab that allows moving around the first two dimensions and viewing the data in the other two dimensions. E.g. for spatial map data, the first two dimensions are usually the spatial co-ordiantes, and the second two dimensions are energy `eV` and angle `theta_par`.
- A region of interest panel, which allows integrating over ROIs defined on either the first two or the second two dimensions. See the video in {ref}`Interactive spatial map explorer <interactive-spatial-map-explorer>` for a demonstration.

In [ ]:
# Static view of the Display Panel for the documentation
plot_tutorial_example_figure("4D_disp_ex1.png")

In [ ]:
# Static view of the Display Panel for the documentation
plot_tutorial_example_figure("4D_disp_ex2.png")

### Inline interactive UIs
A quick version of inline interactive plotting can be achieved by using {py:class}`.iplot <peaks.core.GUI.iplot.hvplot.HVPlotAccessor>` instead of {py:meth}`.plot <xarray.DataArray.plot>`. This is not loaded by default but can be imported if required. It is a very thin wrapper around the holoviews [`hvplot`](https://hvplot.holoviz.org) accessor for {py:mod}`xarray` (see the [`hvplot` documentation](https://hvplot.holoviz.org/user_guide/index.html) and the [`xarray` tutorial](https://tutorial.xarray.dev/intermediate/hvplot.html), simply adding a few different default options and handling "dequantify-ing" the array automatically. For e.g. a single dispersion, this gives a simple plot but with interactive controls:

In [ ]:
disp1.iplot(cmap='Blues', y='eV', x='theta_par')

:::{admonition} [`hvplot`](https://hvplot.holoviz.org) display problems
:class: tip

[`hvplot`](https://hvplot.holoviz.org) is known to sometimes have problems displaying output in Jupyter notebooks. In case of problems seeing the output from {py:class}`.iplot <peaks.core.GUI.iplot.hvplot.HVPlotAccessor>`, try running the following code in the notebook:
```python
from bokeh.io import output_notebook
output_notebook()
```
:::

It is also easy to create interactive data explorers by telling {py:class}`iplot <peaks.core.GUI.iplot.hvplot.HVPlotAccessor>` which axes to `groupby`, e.g. for the 4D dataset above, we can make an explorer for the spatial co-ordinates:

In [ ]:
SM1.isel(x1=slice(0,10),x2=slice(0,10)).compute().iplot(groupby=["x1","x2"], y='eV', x='theta_par', cmap='Blues')

### Animations
You can quickly make an animation over some variable using {py:class}`.iplot <peaks.core.GUI.iplot.hvplot.HVPlotAccessor>`, e.g. let's do this for a photon energy-dependent dataset:

In [ ]:
# Load the data
hv1 = ExampleData.hv_map()

In [ ]:
# Make the plot
hv1.bin_spectra(2).iplot(
    groupby='hv',
    widget_type='scrubber',
    widget_location='bottom',
    y='eV',
    x='theta_par',
    cmap='Blues',
    frame_height=300,
    frame_width=500,
)

:::{tip}
The `.bin_spectra(2)` part here is simply $2\times2$ binning the data in energy and angle to reduce the data size for the animation. This is covered in more detail [later](./2_data_processing.ipynb#data-binning).
:::

To make an animation to be saved to disk, can utilise {py:mod}`matplotlib.animation` tools

:::{important}
`ffmpeg` must be installed on the host system. To install via conda: `conda install -c conda-forge ffmpeg`
:::

```python
from matplotlib.animation import FuncAnimation, FFMpegWriter
from tqdm.notebook import tqdm

# Setup plot
fig, ax = plt.subplots(figsize=(6, 4))
img = hv1.isel(hv=0).plot(ax=ax)
title = ax.set_title(f"hv={hv1.hv.values[0]:0.1f} eV")

# Create progress bar
pbar = tqdm(total=len(hv1.hv), desc="Rendering frames")

# Update function
def update(frame):
    img.set_array(hv1.isel(hv=frame).pint.dequantify().values)
    title.set_text(f"hv={hv1.hv.values[frame]:0.1f} eV")
    pbar.update(1)
    return [img, title]

# Create animation
anim = FuncAnimation(fig, update, frames=len(hv1.hv), blit=True)

# Save as MP4
writer = FFMpegWriter(fps=10, metadata={"title": "hv-dep movie"})
anim.save("hv_dep.mp4", writer=writer)

pbar.close()
```